# Correlation Extension — Comprehensive Temporal Correlation Analysis

This notebook investigates **how features relate to each other and to the `GapUp` target** across multiple correlation frameworks.
Designed to complement `temporal_analysis.ipynb` (KarmaLego patterns) and `temporal_data_analysis.ipynb` (cyclic calendar features).

| Section | Technique | Key Question |
|---|---|---|
| 2 | Spearman ρ — feature × GapUp | Which features correlate most with the target? |
| 3 | Spearman matrix + dendrogram | Which feature groups are internally collinear? |
| 4 | ACF / PACF + Ljung-Box | Is GapUp auto-correlated? Do features have memory? |
| 5 | Lagged cross-correlation | Which features lead or lag GapUp by *k* weeks? |
| 6 | Rolling correlation | Do feature–target correlations drift over time? |
| 7 | Year × feature heatmap | Inter-annual regime changes |
| 8 | Quarter / earnings-season | Seasonal shifts in correlations |
| 9 | Granger causality | Do features formally precede GapUp? |
| 10 | Cross-ticker entanglement | Do tickers gap up together (co-movement)? |
| 11 | Partial correlation | Unique feature contribution after controlling for others |
| 12 | Correlation network graph | Visual topology of feature relationships |
| 13 | Summary | Ranked findings + recommendations |

**Data:** `dataset_clean_temporal.csv` — 8,146 primary rows (COVID window excluded), 25 tickers, April 2016 – December 2024.

## 1  Imports & Global Settings

All correlation work uses **Spearman ρ** (not Pearson) because normality tests in `statistical_analysis.ipynb` showed that virtually all features are non-Gaussian.  
Spearman is rank-based, monotone-relationship sensitive, and robust to outliers — ideal for this dataset.

In [ ]:
from __future__ import annotations
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from scipy.stats import spearmanr, pointbiserialr
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

import statsmodels.api as sm
from statsmodels.tsa.stattools import acf, pacf, grangercausalitytests, adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import acorr_ljungbox

try:
    import networkx as nx
    _HAS_NX = True
except ImportError:
    _HAS_NX = False
    print('[warn] networkx not found — Section 12 will be skipped')

plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 10,
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'figure.facecolor': 'white',
})
BLUE = '#4c8af5'; RED = '#f55c5c'; GREY = '#aaaaaa'
DIV_CMAP = 'RdBu_r'
SEQ_CMAP = 'YlOrRd'
print('Imports OK')


## 2  Load Data & Define Feature Groups

We use `dataset_clean_temporal.csv` (adds the 7 cyclic calendar features) and apply the same primary-set filter as all other notebooks: `is_extreme_event == 0`.

Features are split into six groups:
- **Momentum** — RSI, MACD, ROC, StochPercK, CloseVEma50, CloseVSma20
- **Volatility** — ADX, BollingerBandWidth, ATR, FiveDStdDev, IntraWeekVolatility, WeeklyRange
- **Volume** — OBV, MFI, VolumeRatio
- **PriceMicro** — WeeklyReturn, FridayPosition, OpenCloseSpread, GapMagnitude
- **Fundamental** — NetMargin, RoA, RevGrowthQoQ, GrossMargin
- **Temporal** — Month_sin/cos, WeekOfYear_sin/cos, Quarter_sin/cos, DaysSinceStart

In [ ]:
PROJECT_ROOT  = Path('..').resolve()
TEMPORAL_PATH = PROJECT_ROOT / 'structured_csv_data_files' / 'fetched_data' / 'dataset_clean_temporal.csv'
TARGET = 'GapUp'

df_all  = pd.read_csv(TEMPORAL_PATH)
df_all['Date'] = pd.to_datetime(df_all['Date'], utc=True).dt.tz_convert(None)
primary = df_all[df_all['is_extreme_event'] == 0].copy()
primary = primary.sort_values(['Ticker', 'Date']).reset_index(drop=True)

print(f'Primary rows : {len(primary):,}  ({primary["Ticker"].nunique()} tickers, '
      f'{primary["Year"].nunique()} years)')
print(f'GapUp rate   : {primary[TARGET].mean():.4f}')
print(f'Date range   : {primary["Date"].min().date()} -> {primary["Date"].max().date()}')

MOMENTUM    = ['RSI','MACD','ROC','StochPercK','CloseVEma50','CloseVSma20']
VOLATILITY  = ['ADX','BollingerBandWidth','ATR','FiveDStdDev','IntraWeekVolatility','WeeklyRange']
VOLUME      = ['OBV','MFI','VolumeRatio']
PRICE_MICRO = ['WeeklyReturn','FridayPosition','OpenCloseSpread','GapMagnitude']
FUNDAMENTAL = ['NetMargin','RoA','RevGrowthQoQ','GrossMargin']
TEMPORAL_F  = ['Month_sin','Month_cos','WeekOfYear_sin','WeekOfYear_cos',
                'Quarter_sin','Quarter_cos','DaysSinceStart']

ALL_FEATURES = MOMENTUM + VOLATILITY + VOLUME + PRICE_MICRO + FUNDAMENTAL + TEMPORAL_F
BASELINE     = ['MACD','ROC','StochPercK','CloseVEma50','CloseVSma20','ADX',
                 'BollingerBandWidth','ATR','FiveDStdDev','OBV','MFI','VolumeRatio',
                 'NetMargin','RoA','RevGrowthQoQ']

GROUP_MAP = {f: g for g, fl in [
    ('Momentum',   MOMENTUM), ('Volatility', VOLATILITY),
    ('Volume',     VOLUME),   ('PriceMicro', PRICE_MICRO),
    ('Fundamental',FUNDAMENTAL),('Temporal',  TEMPORAL_F),
] for f in fl}

GROUP_COLORS = {
    'Momentum':'#4c8af5','Volatility':'#f5a623','Volume':'#7ed321',
    'PriceMicro':'#9b59b6','Fundamental':'#e74c3c','Temporal':'#1abc9c',
}

X = primary[ALL_FEATURES].copy()
y = primary[TARGET].values
print(f'Feature matrix: {X.shape}  |  Groups: {list(GROUP_COLORS.keys())}')


## 3  Spearman ρ — Every Feature vs GapUp

**Spearman ρ** measures monotone rank association between a continuous feature and the binary `GapUp` target.  For a binary target this is rank-equivalent to point-biserial correlation.

We also compute the **p-value** (H₀: ρ = 0) and flag features significant at α = 0.05 with **Bonferroni correction** for the number of tests.

> Bars are coloured by feature group. Faded bars are not significant after correction.  
> Grey dashed lines mark the ±0.05 practical significance threshold.

In [ ]:
alpha_bonf = 0.05 / len(ALL_FEATURES)

rows = []
for feat in ALL_FEATURES:
    vals = primary[feat].values.astype(float)
    rho, pval = spearmanr(vals, y, nan_policy='omit')
    rows.append({'Feature': feat, 'rho': rho, 'pval': pval,
                 'Group': GROUP_MAP[feat],
                 'Significant': pval < alpha_bonf})

spear_df = pd.DataFrame(rows).sort_values('rho', key=abs, ascending=False)
print(f'Bonferroni alpha = {alpha_bonf:.4f}')
print(f'Significant features: {spear_df["Significant"].sum()} / {len(spear_df)}')
print()
print(spear_df[['Feature','Group','rho','pval','Significant']].to_string(index=False))


In [ ]:
from matplotlib.patches import Patch

fig, ax = plt.subplots(figsize=(14, 8))
df_plot = spear_df.copy().reset_index(drop=True)
colors  = [GROUP_COLORS[g] for g in df_plot['Group']]

bars = ax.barh(df_plot['Feature'], df_plot['rho'],
               color=colors, edgecolor='white', height=0.75)
for bar, sig in zip(bars, df_plot['Significant']):
    bar.set_alpha(0.90 if sig else 0.30)

ax.axvline(0,     color='black', linewidth=0.8)
ax.axvline( 0.05, color='grey',  linewidth=0.6, linestyle='--', alpha=0.5)
ax.axvline(-0.05, color='grey',  linewidth=0.6, linestyle='--', alpha=0.5)
ax.set_xlabel('Spearman rho  (feature vs GapUp)')
ax.set_title('Spearman Correlation with GapUp — All Features\n'
             '(faded = not significant after Bonferroni correction)')
ax.invert_yaxis()

handles = [Patch(color=c, label=g) for g, c in GROUP_COLORS.items()]
ax.legend(handles=handles, loc='lower right', fontsize=8, framealpha=0.9)
plt.tight_layout()
plt.show()


## 4  Feature–Feature Spearman Correlation Matrix

Full pairwise Spearman correlation matrix, reordered by **hierarchical clustering** (Ward linkage on correlation-distance `1 − |ρ|`).

- Distance = 0 → features are perfectly correlated (|ρ| = 1) → identical signal.
- Distance = 1 → features are orthogonal (|ρ| = 0) → carry completely independent information.

Ward linkage minimises total within-cluster variance, grouping the most similar features into tight clusters first.

This extends the **Spearman** correlation analysis in `statistical_analysis.ipynb` Section 3 — the difference here is hierarchical clustering reordering and explicit feature-group annotation on the axes.

> **How to read this plot:**  
> — **Axis tick labels** are **colour-coded by feature group** (blue=Momentum, orange=Volatility, green=Volume, purple=PriceMicro, red=Fundamental, teal=Temporal).  
> — **Deep red blocks** = strong positive co-movement; **deep blue blocks** = strong negative relationship.  
> — Features in the same colour-coded tick cluster that also form a tight dendrogram branch are prime candidates for **VIF pruning** or **PCA compression** — they carry near-identical signal.

We use `sns.clustermap()` with the precomputed Ward linkage matrix, which correctly aligns the dendrogram leaves with the heatmap rows — avoiding the top/bottom reversal that occurs when positioning a dendrogram and heatmap in separate matplotlib axes.

In [ ]:
# Compute full pairwise Spearman correlation matrix
# We iterate manually to exploit symmetry (rho(f1,f2) = rho(f2,f1))
# so we only call spearmanr for the upper triangle.
corr_vals = {f1: {} for f1 in ALL_FEATURES}
for i, f1 in enumerate(ALL_FEATURES):
    corr_vals[f1][f1] = 1.0
    for j, f2 in enumerate(ALL_FEATURES):
        if j <= i:
            continue
        r, _ = spearmanr(primary[f1].values, primary[f2].values, nan_policy='omit')
        corr_vals[f1][f2] = r
        corr_vals[f2][f1] = r   # symmetric

corr_matrix = pd.DataFrame(corr_vals, index=ALL_FEATURES, columns=ALL_FEATURES)

# Compute Ward linkage on distance 1 - |rho|
# squareform(NxN_matrix) converts it to a condensed 1-D distance array for linkage().
# The diagonal (self-distance = 0) is NOT included in the condensed form,
# so np.fill_diagonal is neither needed nor valid on the 1-D output.
dist = squareform(1 - corr_matrix.abs().values, checks=False)
Z       = linkage(dist, method='ward')
order   = dendrogram(Z, no_plot=True)['leaves']
feat_ord = [ALL_FEATURES[i] for i in order]

# Reorder matrix to match clustering (both rows and cols)
corr_ord = corr_matrix.loc[feat_ord, feat_ord]

# Report highly collinear pairs
print('Highly collinear pairs  |rho| > 0.70:')
found = False
for f1, f2 in combinations(ALL_FEATURES, 2):
    r = corr_matrix.loc[f1, f2]
    if abs(r) > 0.70:
        print(f'  {f1:>22s} <-> {f2:<22s}  rho={r:+.3f}')
        found = True
if not found:
    print('  None found above threshold.')

In [ ]:
import numpy as np

# Lower-triangle mask: hide upper triangle (it is redundant in a symmetric matrix)
mask = np.triu(np.ones_like(corr_ord.values, dtype=bool), k=1)

# Colour lookup for tick labels by group
row_colors = pd.Series(
    [GROUP_COLORS[GROUP_MAP[f]] for f in feat_ord],
    index=feat_ord, name='Group'
)

# sns.clustermap accepts a precomputed linkage via row_linkage / col_linkage.
# This guarantees dendrogram leaf order == heatmap row/col order (no reversal bug).
g = sns.clustermap(
    corr_ord,
    row_linkage=Z, col_linkage=Z,
    cmap=DIV_CMAP, center=0, vmin=-1, vmax=1,
    linewidths=0.25, linecolor='#dddddd',
    mask=mask,
    row_colors=row_colors, col_colors=row_colors,
    figsize=(18, 16),
    cbar_pos=(0.02, 0.83, 0.03, 0.14),
    dendrogram_ratio=0.12,
    xticklabels=True, yticklabels=True,
)

# Colour each tick label by its feature group
g.ax_heatmap.figure.canvas.draw()   # flush so labels are populated
for tick in g.ax_heatmap.get_xticklabels():
    feat_name = tick.get_text()
    tick.set_color(GROUP_COLORS.get(GROUP_MAP.get(feat_name, ''), 'black'))
    tick.set_fontsize(8)
    tick.set_rotation(90)
for tick in g.ax_heatmap.get_yticklabels():
    feat_name = tick.get_text()
    tick.set_color(GROUP_COLORS.get(GROUP_MAP.get(feat_name, ''), 'black'))
    tick.set_fontsize(8)

g.ax_heatmap.set_title(
    'Feature-Feature Spearman Correlation Matrix\n'
    '(lower triangle; rows/cols reordered by Ward clustering on 1-|rho|)\n'
    'Tick label colour = feature group (see Section 2 legend)',
    pad=12, fontsize=11
)

# Group legend
from matplotlib.patches import Patch
handles = [Patch(color=c, label=g_name) for g_name, c in GROUP_COLORS.items()]
g.ax_heatmap.legend(
    handles=handles, title='Feature group',
    loc='lower left', bbox_to_anchor=(-0.6, 0),
    fontsize=8, framealpha=0.9
)

plt.show()


## 5  Autocorrelation — ACF, PACF & Ljung-Box Test

**ACF (AutoCorrelation Function):** Spearman correlation of the series with itself at lag *k*.  
**PACF (Partial ACF):** correlation at lag *k* after removing the linear influence of lags 1 through *k−1* — reveals the *direct* memory structure.  
**Ljung-Box test:** joint hypothesis test that autocorrelations up to lag *m* are all zero.

Three analyses, each solving a different problem:

| Analysis | Series used | Purpose |
|---|---|---|
| GapUp ACF / PACF | Cross-sectional **weekly median** across all tickers | Is the *target* itself serially correlated? |
| Top-6 feature ACF | Cross-sectional **weekly median** | Do strong predictors carry multi-week momentum? |
| Per-ticker Ljung-Box | **Each ticker separately** | How many tickers show significant autocorrelation at each lag? |

> **Why weekly median instead of pooling raw rows?**  
> Pooling all 25 tickers' raw rows sorted by date creates multiple observations per week.  
> This artificially inflates the series length and destroys the temporal ordering — ACF values become meaningless.  
> The cross-sectional median gives one representative observation per calendar week.

> **Interpretation benchmark:** A Durbin-Watson statistic near 2.0 and Ljung-Box p > 0.05 for `GapUp` at lag 1 confirms the target is largely white noise week-to-week — validating the cross-sectional model approach (LogReg/XGBoost). Features with high Ljung-Box counts carry multi-week momentum that LSTM models (`lstm_model.ipynb`) can exploit.

In [ ]:
# Use per-date cross-sectional median to get one value per week
# This avoids the multiple-tickers-per-date pooling problem
weekly_median = primary.groupby('Date')[ALL_FEATURES + [TARGET]].median().sort_index()
gap_series    = weekly_median[TARGET].values

print(f'Weekly median series length: {len(gap_series)} weeks  '
      f'(vs {len(primary)} raw rows if pooled)')

fig, axes = plt.subplots(2, 1, figsize=(14, 7))
plot_acf(gap_series,  lags=min(26, len(gap_series)//2 - 1), alpha=0.05, ax=axes[0],
         title='ACF — GapUp  (cross-sectional weekly median, all tickers)')
plot_pacf(gap_series, lags=min(26, len(gap_series)//2 - 1), alpha=0.05,
          ax=axes[1], method='ywm',
          title='PACF — GapUp  (cross-sectional weekly median)')
for ax in axes:
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xlabel('Lag (weeks)')
    ax.set_ylabel('Autocorrelation')
plt.tight_layout()
plt.show()

dw = durbin_watson(gap_series)
print(f'Durbin-Watson: {dw:.4f}  (2.0 = no autocorrelation, <2 = positive, >2 = negative)')

# Ljung-Box at lag 1 for the weekly median GapUp
lb1 = acorr_ljungbox(gap_series, lags=[1, 4, 8], return_df=True)
print('\nLjung-Box on weekly median GapUp:')
print(lb1.to_string())


In [ ]:
# Top-6 features by |Spearman rho| with GapUp
# Use weekly median series (same approach as GapUp above)
top6_feats = spear_df['Feature'].head(6).tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(top6_feats):
    # Weekly median — one value per calendar week, no per-ticker mixing
    series  = weekly_median[feat].values
    rho_val = spear_df.loc[spear_df['Feature'] == feat, 'rho'].values[0]
    n_lags  = min(20, len(series) // 2 - 1)
    plot_acf(series, lags=n_lags, alpha=0.05, ax=axes[i],
             title=f'ACF: {feat}  (rho_GapUp={rho_val:+.3f})')
    axes[i].set_xlabel('Lag (weeks)')
    axes[i].set_ylabel('ACF')

plt.suptitle('ACF of Top-6 Features by |Spearman rho| with GapUp\n'
             '(cross-sectional weekly median; blue band = 95% CI under white-noise null)',
             y=1.02)
plt.tight_layout()
plt.show()

# ── Bonus: single representative ticker (ADBE) raw ACF for comparison ────
rep_ticker = 'ADBE'
rep_data   = primary[primary['Ticker'] == rep_ticker].sort_values('Date')

fig2, axes2 = plt.subplots(2, 3, figsize=(16, 8))
axes2 = axes2.flatten()
for i, feat in enumerate(top6_feats):
    series2 = rep_data[feat].values
    rho_val = spear_df.loc[spear_df['Feature'] == feat, 'rho'].values[0]
    plot_acf(series2, lags=min(20, len(series2)//2 - 1), alpha=0.05, ax=axes2[i],
             title=f'ACF: {feat}  ({rep_ticker}, rho={rho_val:+.3f})')
    axes2[i].set_xlabel('Lag (weeks)')

plt.suptitle(f'ACF of Top-6 Features — Single Ticker ({rep_ticker})\n'
             '(per-entity view; compare with cross-sectional median above)',
             y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
lags_to_test = [1, 2, 4, 8, 13, 26]
lb_rows = []

for feat in BASELINE + [TARGET]:
    n_sig = {lag: 0 for lag in lags_to_test}
    for ticker, grp in primary.groupby('Ticker'):
        # Per-ticker series — sort chronologically, forward-fill any gaps
        # Use .ffill() instead of fillna(method='ffill') (deprecated in pandas 2.x)
        series = grp.sort_values('Date')[feat].ffill().values
        if len(series) < 30:
            continue
        lb_res = acorr_ljungbox(series, lags=lags_to_test, return_df=True)
        for lag in lags_to_test:
            if lb_res.loc[lag, 'lb_pvalue'] < 0.05:
                n_sig[lag] += 1
    row = {'Feature': feat}
    row.update({f'lag_{l}': n_sig[l] for l in lags_to_test})
    lb_rows.append(row)

lb_df = pd.DataFrame(lb_rows).set_index('Feature')

# Sort: features where autocorrelation is most common at short lags appear first
lb_df = lb_df.sort_values('lag_1', ascending=False)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    lb_df, ax=ax,
    cmap='YlOrRd', annot=True, fmt='d',
    cbar_kws={'label': '# tickers with significant autocorrelation (out of 25)'},
    linewidths=0.5, linecolor='white',
    vmin=0, vmax=25
)
ax.set_title(
    'Ljung-Box Test — Per-Ticker Significant Autocorrelation Count\n'
    '(cells show # of tickers with p < 0.05 at each lag; max = 25)\n'
    'Sorted by lag-1 count descending — features with persistent memory appear at top',
    fontsize=10
)
ax.set_xlabel('Lag (weeks)')
ax.set_ylabel('Feature')

# Add group colour bar on y-axis
for tick in ax.get_yticklabels():
    feat_name = tick.get_text()
    tick.set_color(GROUP_COLORS.get(GROUP_MAP.get(feat_name, ''), 'black'))

plt.tight_layout()
plt.show()

print('\nFeatures where ALL 25 tickers show autocorrelation at lag 1:')
print(lb_df[lb_df['lag_1'] == 25].index.tolist())
print('\nFeatures where NO tickers show autocorrelation at lag 1:')
print(lb_df[lb_df['lag_1'] == 0].index.tolist())

## 6  Lagged Cross-Correlation — Lead / Lag Analysis

For each feature at time *t*, compute Spearman ρ with `GapUp` at time *t + k* (k = −6 to +6 weeks).  This is the core **temporal entanglement** diagnostic:

| Lag direction | Meaning |
|---|---|
| **k > 0** (feature leads) | Feature at *t* predicts GapUp at *t+k* — **actionable predictor** |
| **k = 0** | Same-week association — valid if feature is observable before market open |
| **k < 0** (feature lags) | GapUp at *t* causes feature change at *t+k* — potential leakage |

Computed **per ticker** then averaged to prevent any single ticker from dominating.

In [ ]:
LAGS = list(range(-6, 7))

def lagged_spearman_per_ticker(feat, target, lag, df):
    rhos = []
    for _, grp in df.groupby('Ticker'):
        grp = grp.sort_values('Date').reset_index(drop=True)
        x = grp[feat].values.astype(float)
        y_shifted = grp[target].shift(-lag).values
        mask = ~np.isnan(x) & ~np.isnan(y_shifted)
        if mask.sum() < 20:
            continue
        r, _ = spearmanr(x[mask], y_shifted[mask])
        rhos.append(r)
    return float(np.nanmean(rhos)) if rhos else np.nan

print('Computing lagged cross-correlations ...')
lag_records = {}
for feat in ALL_FEATURES:
    lag_records[feat] = {lag: lagged_spearman_per_ticker(feat, TARGET, lag, primary)
                         for lag in LAGS}

lag_df = pd.DataFrame(lag_records).T
lag_df.columns = [f'lag_{l:+d}' for l in LAGS]
lag_df['_sort'] = lag_df['lag_+1'].abs()
lag_df = lag_df.sort_values('_sort', ascending=False).drop(columns='_sort')
print('Done.')
print(lag_df.round(3).to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 10))

# Left heatmap
sns.heatmap(
    lag_df, ax=axes[0],
    cmap=DIV_CMAP, center=0, vmin=-0.15, vmax=0.15,
    linewidths=0.4, linecolor='white',
    annot=True, fmt='.2f', annot_kws={'size': 7},
    cbar_kws={'label': 'Spearman rho'},
)
axes[0].set_title('Lagged Cross-Correlation: Feature at t  vs  GapUp at t+lag\n'
                  '(positive lag = feature leads GapUp)')
axes[0].axvline(LAGS.index(0) + 0.5, color='black', lw=2, linestyle='--')
axes[0].set_xlabel('Lag (weeks)')

# Right line plot — top 5
top5 = lag_df.index[:5].tolist()
ax2  = axes[1]
for feat in top5:
    vals = lag_df.loc[feat].values
    ax2.plot(LAGS, vals, marker='o', markersize=4, label=feat, linewidth=1.8)
ax2.axhline(0, color='black', lw=0.8)
ax2.axvline(0, color='grey',  lw=0.8, linestyle='--')
ax2.axvline(1, color='red',   lw=0.8, linestyle=':', alpha=0.7)
ax2.set_xlabel('Lag k  (feature at t vs GapUp at t+k)')
ax2.set_ylabel('Mean Spearman rho  (across tickers)')
ax2.set_title('Lead-Lag Profile — Top 5 Features\n'
              '(red dotted = lag +1, most actionable prediction horizon)')
ax2.legend(fontsize=8)
ax2.set_xticks(LAGS)

plt.tight_layout()
plt.show()


## 7  Rolling Correlation — Temporal Drift Detection

A **52-week rolling Spearman correlation** (cross-sectional median series) shows whether the feature–target relationship is stable or regime-dependent.

- **Stable, consistently-signed** rolling correlation → reliable predictor across market regimes.
- **Sign-flipping** or high-variance rolling correlation → regime-dependent, use with caution.
- Shaded bands mark calendar years for easy regime identification.

> The 2020 COVID year often shows anomalous correlation spikes — a key reason the primary set excludes the Feb–May 2020 window.

In [ ]:
weekly = primary.groupby('Date')[ALL_FEATURES + [TARGET]].median().sort_index()
WINDOW = 52
top4   = spear_df['Feature'].head(4).tolist()

def rolling_spearman(s_feat, s_tgt, window):
    """True rolling Spearman rho — pandas rolling().corr() only does Pearson."""
    vals = []
    for i in range(len(s_feat)):
        if i < window - 1:
            vals.append(np.nan)
        else:
            sl_x = s_feat.iloc[i - window + 1 : i + 1]
            sl_y = s_tgt.iloc[i - window + 1 : i + 1]
            ok   = sl_x.notna() & sl_y.notna()
            if ok.sum() < 10:
                vals.append(np.nan)
            else:
                r, _ = spearmanr(sl_x[ok], sl_y[ok])
                vals.append(r)
    return pd.Series(vals, index=s_feat.index)

fig, axes = plt.subplots(len(top4), 1, figsize=(15, 11), sharex=True)
year_starts = [pd.Timestamp(f'{y}-01-01') for y in range(2016, 2026)]

for ax, feat in zip(axes, top4):
    roll_rho = rolling_spearman(weekly[feat], weekly[TARGET], WINDOW)
    ax.plot(weekly.index, roll_rho,
            color=GROUP_COLORS[GROUP_MAP[feat]], linewidth=1.8, label=feat)
    ax.fill_between(weekly.index, roll_rho, 0,
                    where=roll_rho > 0, alpha=0.15, color='green')
    ax.fill_between(weekly.index, roll_rho, 0,
                    where=roll_rho < 0, alpha=0.15, color='red')
    for i in range(len(year_starts) - 1):
        ax.axvspan(year_starts[i], year_starts[i+1],
                   alpha=0.04, color='blue' if i % 2 == 0 else 'grey')
    ax.axhline(0,     color='black', lw=0.7)
    ax.axhline( 0.10, color='grey',  lw=0.5, linestyle=':')
    ax.axhline(-0.10, color='grey',  lw=0.5, linestyle=':')
    ax.set_ylabel('rho (52-wk roll.)')
    ax.set_ylim(-0.55, 0.55)
    ax.legend(loc='upper left', fontsize=9)

axes[-1].set_xlabel('Date')
axes[0].set_title(f'Rolling {WINDOW}-Week Spearman Correlation with GapUp\n'
                  '(cross-sectional weekly median across all tickers)')
plt.tight_layout()
plt.show()

## 8  Inter-Annual Correlation Drift — Year × Feature Heatmap

Spearman ρ (feature vs GapUp) computed **separately for each calendar year**.  This reveals:

- **Consistent features** — same sign and similar magnitude every year → reliable.
- **Year-specific spikes** — may be spurious or regime-specific.
- **COVID anomaly (2020)** — highlighted with an orange border; correlations that year are unreliable for general modelling.

Sorted by mean |ρ| across years — most reliable predictors appear at the top.

In [ ]:
years = sorted(primary['Year'].unique())
year_corr = {}
for yr in years:
    yr_data = primary[primary['Year'] == yr]
    year_corr[yr] = {
        feat: spearmanr(yr_data[feat].values, yr_data[TARGET].values, nan_policy='omit')[0]
        for feat in ALL_FEATURES
    }

yc_df = pd.DataFrame(year_corr)
yc_df['mean_abs'] = yc_df.abs().mean(axis=1)
yc_df = yc_df.sort_values('mean_abs', ascending=False).drop(columns='mean_abs')

fig, ax = plt.subplots(figsize=(13, 12))
sns.heatmap(
    yc_df, ax=ax,
    cmap=DIV_CMAP, center=0, vmin=-0.30, vmax=0.30,
    annot=True, fmt='.2f', annot_kws={'size': 8},
    linewidths=0.4, linecolor='white',
    cbar_kws={'label': 'Spearman rho'},
)

# Highlight COVID year
covid_col = list(yc_df.columns).index(2020)
ax.add_patch(plt.Rectangle((covid_col, 0), 1, len(yc_df),
             fill=False, edgecolor='orange', lw=3, clip_on=False))
ax.text(covid_col + 0.5, -0.6, 'COVID', ha='center', color='orange', fontsize=8)

ax.set_title('Inter-Annual Correlation Drift\n'
             'Spearman rho (feature vs GapUp) per calendar year\n'
             '(sorted by mean |rho|; orange = COVID year)')
ax.set_xlabel('Year')
plt.tight_layout()
plt.show()


## 9  Seasonal Correlation — Quarter & Earnings-Week Breakdown

Do feature–target correlations shift across **quarters** or **earnings windows**?  
We slice by quarter (Q1–Q4) and by the earnings-week flag from `temporal_extensions.ipynb` (ISO weeks 1–5, 14–18, 27–31, 40–44).

If correlations are stronger in earnings windows, those periods may warrant separate model treatment.

In [ ]:
quarters = ['Q1','Q2','Q3','Q4']
q_corr   = {q: {feat: spearmanr(primary[primary['Quarter']==q][feat],
                                  primary[primary['Quarter']==q][TARGET],
                                  nan_policy='omit')[0]
                 for feat in BASELINE}
             for q in quarters}
qc_df = pd.DataFrame(q_corr)

earnings_weeks = set(range(14,19))|set(range(27,32))|set(range(40,45))|set(range(1,6))|{53}
prim_earn    = primary[ primary['WeekOfYear'].isin(earnings_weeks)]
prim_notearn = primary[~primary['WeekOfYear'].isin(earnings_weeks)]

earn_rho    = [spearmanr(prim_earn[f],    prim_earn[TARGET],    nan_policy='omit')[0] for f in BASELINE]
notearn_rho = [spearmanr(prim_notearn[f], prim_notearn[TARGET], nan_policy='omit')[0] for f in BASELINE]

fig, axes = plt.subplots(1, 2, figsize=(17, 7))

sns.heatmap(qc_df, ax=axes[0], cmap=DIV_CMAP, center=0, vmin=-0.25, vmax=0.25,
            annot=True, fmt='.2f', annot_kws={'size': 9},
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Spearman rho'})
axes[0].set_title('Spearman rho with GapUp by Quarter')

x = np.arange(len(BASELINE))
w = 0.35
axes[1].bar(x - w/2, earn_rho,    w, label='Earnings weeks',     color=RED,  alpha=0.8)
axes[1].bar(x + w/2, notearn_rho, w, label='Non-earnings weeks', color=BLUE, alpha=0.8)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(BASELINE, rotation=50, ha='right', fontsize=8)
axes[1].set_ylabel('Spearman rho')
axes[1].set_title('Correlation vs GapUp: Earnings vs Non-Earnings Weeks')
axes[1].legend()

plt.tight_layout()
plt.show()


## 10  Granger Causality — Does Any Feature Formally Precede GapUp?

**Granger causality** tests whether past values of feature *X* improve the prediction of `GapUp` *beyond* what past `GapUp` values alone can predict.  It measures **predictive precedence** in a linear VAR sense — not true causal inference.

Test run on the cross-sectional weekly median series (one time series per feature), at lag orders 1, 2, and 4.  We first confirm stationarity with the **ADF test** (non-stationary features cannot be validly Granger-tested without differencing).

> Green borders in the heatmap mark cells with p < 0.05.  Annotation shows the p-value; cell colour shows the F-statistic.

In [ ]:
gc_results = []
lag_orders  = [1, 2, 4]

for feat in BASELINE + TEMPORAL_F:
    ts = primary.groupby('Date')[[feat, TARGET]].median().sort_index().dropna()
    x_col = ts[feat].values
    y_col = ts[TARGET].values
    adf_p = adfuller(x_col, autolag='AIC')[1]
    combined = np.column_stack([y_col, x_col])
    for lag in lag_orders:
        try:
            gc_out = grangercausalitytests(combined, maxlag=lag, verbose=False)
            fstat  = gc_out[lag][0]['ssr_ftest'][0]
            pval   = gc_out[lag][0]['ssr_ftest'][1]
        except Exception:
            fstat, pval = np.nan, np.nan
        gc_results.append({
            'Feature': feat, 'lag': lag,
            'F-stat': fstat, 'p-value': pval,
            'Stationary': adf_p < 0.05,
            'Significant': (not np.isnan(pval)) and pval < 0.05,
        })

gc_df = pd.DataFrame(gc_results)
print('Significant Granger-causal features (p < 0.05):')
sig = gc_df[gc_df['Significant']]
print(sig[['Feature','lag','F-stat','p-value','Stationary']].round(4).to_string(index=False))


In [ ]:
pivot_f = gc_df.pivot(index='Feature', columns='lag', values='F-stat')
pivot_p = gc_df.pivot(index='Feature', columns='lag', values='p-value')
pivot_f = pivot_f.loc[pivot_f.max(axis=1).sort_values(ascending=False).index]
pivot_p = pivot_p.loc[pivot_f.index]

fig, ax = plt.subplots(figsize=(8, 11))
sns.heatmap(
    pivot_f.fillna(0), ax=ax,
    cmap=SEQ_CMAP,
    annot=pivot_p.round(3).fillna(1), fmt='',
    annot_kws={'size': 8},
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'F-statistic'},
)
for i, feat in enumerate(pivot_f.index):
    for j, lag in enumerate(pivot_f.columns):
        pv = pivot_p.loc[feat, lag]
        if not np.isnan(pv) and pv < 0.05:
            ax.add_patch(plt.Rectangle((j, i), 1, 1,
                         fill=False, edgecolor='lime', lw=2.5))

ax.set_title('Granger Causality F-statistic  (Feature -> GapUp)\n'
             'Green border = p < 0.05  |  Annotation = p-value')
ax.set_xlabel('Lag order (weeks)')
plt.tight_layout()
plt.show()


## 11  Cross-Ticker Temporal Entanglement

**Temporal entanglement** measures whether multiple tickers tend to gap up or down *at the same time*.

**Why it matters:**
1. High cross-ticker synchronisation → observations are not independent → effective sample size < row count → models may overfit.
2. Clusters of synchronised tickers reveal sector-level momentum that could be exploited or hedged.

**Method:** Pivot to a date × ticker GapUp matrix.  Compute pairwise Spearman ρ between ticker time-series.  Cluster the 25×25 matrix.  Also plot the **rolling 26-week mean pairwise correlation** to see synchronisation regimes.

In [ ]:
pivot_gap = primary.pivot_table(index='Date', columns='Ticker', values='GapUp')
pivot_gap = pivot_gap.sort_index().dropna(thresh=20, axis=1)
tickers   = pivot_gap.columns.tolist()

n = len(tickers)
tc_vals = np.eye(n)
for i, t1 in enumerate(tickers):
    for j, t2 in enumerate(tickers):
        if i < j:
            s = pivot_gap[[t1, t2]].dropna()
            r, _ = spearmanr(s[t1], s[t2])
            tc_vals[i, j] = tc_vals[j, i] = r

ticker_corr = pd.DataFrame(tc_vals, index=tickers, columns=tickers)
off_diag    = tc_vals[np.triu_indices(n, k=1)]
print(f'Mean off-diagonal Spearman rho: {off_diag.mean():.4f}')
print(f'Std: {off_diag.std():.4f}  Min: {off_diag.min():.4f}  Max: {off_diag.max():.4f}')

# squareform(NxN) → condensed 1-D distance array; diagonal is excluded from condensed form
# so np.fill_diagonal is neither needed nor valid here
tc_dist  = squareform(1 - ticker_corr.abs().values, checks=False)
tc_Z     = linkage(tc_dist, method='ward')
tc_order = dendrogram(tc_Z, no_plot=True)['leaves']
tick_ord = [tickers[i] for i in tc_order]
tc_ord   = ticker_corr.loc[tick_ord, tick_ord]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Left: ticker heatmap
sns.heatmap(tc_ord, ax=axes[0], cmap=DIV_CMAP, center=0, vmin=-0.4, vmax=0.4,
            annot=True, fmt='.2f', annot_kws={'size': 6.5},
            linewidths=0.4, linecolor='white',
            cbar_kws={'label': 'Spearman rho  (GapUp series)'})
axes[0].set_title('Cross-Ticker GapUp Correlation Matrix\n'
                  '(clustered — warm = tend to gap up together)')

# Right: rolling mean pairwise correlation
ROLL = 26
dates_all = pivot_gap.index
roll_vals = []
for i, dt in enumerate(dates_all):
    start = max(0, i - ROLL + 1)
    sl    = pivot_gap.iloc[start:i+1]
    if len(sl) < 10:
        roll_vals.append(np.nan); continue
    vals = []
    for t1, t2 in combinations(sl.columns, 2):
        s = sl[[t1, t2]].dropna()
        if len(s) > 8:
            r, _ = spearmanr(s[t1], s[t2])
            vals.append(r)
    roll_vals.append(np.nanmean(vals) if vals else np.nan)

roll_sync = pd.Series(roll_vals, index=dates_all)
ax2 = axes[1]
ax2.fill_between(roll_sync.index, roll_sync, alpha=0.3, color=RED)
ax2.plot(roll_sync.index, roll_sync, color=RED, lw=1.5)
ax2.axhline(0, color='black', lw=0.8)
ax2.axhline(roll_sync.mean(), color='grey', lw=1, linestyle='--',
            label=f'Mean = {roll_sync.mean():.3f}')
ax2.set_title(f'Rolling {ROLL}-Week Mean Pairwise GapUp Correlation\n(all 25 ticker pairs)')
ax2.set_xlabel('Date')
ax2.set_ylabel('Mean Spearman rho')
ax2.legend(fontsize=9)
plt.tight_layout()
plt.show()


## 12  Partial Correlation — Unique Feature Contribution

**Partial correlation** measures association between a feature and `GapUp` **after controlling for all other features** in the baseline set.  It removes shared variance explained by confounders.

- High raw ρ + near-zero partial ρ → **redundant** feature (captured by others).
- High raw ρ + high partial ρ → **uniquely informative** feature.

**Method:** Rank-based OLS residuals (semipartial approach) — rank all variables, regress each feature and `GapUp` on the remaining features, then correlate the residuals.  This is the Spearman analogue of standard partial correlation.

In [ ]:
from numpy.linalg import lstsq

def partial_spearman(df, target, features):
    rank_df = df[features + [target]].dropna().rank()
    y_rank  = rank_df[target].values
    partials = {}
    for feat in features:
        others = [f for f in features if f != feat]
        X_ctrl = rank_df[others].values
        X_ctrl = np.hstack([np.ones((len(X_ctrl), 1)), X_ctrl])
        coef_x, _, _, _ = lstsq(X_ctrl, rank_df[feat].values, rcond=None)
        res_x = rank_df[feat].values - X_ctrl @ coef_x
        coef_y, _, _, _ = lstsq(X_ctrl, y_rank, rcond=None)
        res_y = y_rank - X_ctrl @ coef_y
        r, _  = spearmanr(res_x, res_y)
        partials[feat] = r
    return pd.Series(partials)

print('Computing partial Spearman correlations ...')
partial_rho = partial_spearman(primary, TARGET, BASELINE)

compare_df = pd.DataFrame({
    'Raw Spearman rho':     spear_df.set_index('Feature').loc[BASELINE, 'rho'],
    'Partial Spearman rho': partial_rho,
}).sort_values('Raw Spearman rho', key=abs, ascending=False)

print(compare_df.round(4).to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(compare_df))
w = 0.38
ax.bar(x - w/2, compare_df['Raw Spearman rho'],     w,
       label='Raw Spearman rho',     color=BLUE, alpha=0.85, edgecolor='white')
ax.bar(x + w/2, compare_df['Partial Spearman rho'], w,
       label='Partial Spearman rho', color=RED,  alpha=0.85, edgecolor='white')
ax.axhline(0, color='black', lw=0.8)
ax.axhline( 0.03, color='grey', lw=0.5, linestyle=':')
ax.axhline(-0.03, color='grey', lw=0.5, linestyle=':')
ax.set_xticks(x)
ax.set_xticklabels(compare_df.index, rotation=50, ha='right', fontsize=9)
ax.set_ylabel('Spearman rho')
ax.set_title('Raw vs Partial Spearman Correlation with GapUp\n'
             '(partial controls for all other 14 baseline features)')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()


## 13  Feature Correlation Network Graph

Network topology of feature relationships:
- **Nodes**: coloured by feature group, sized by |Spearman ρ with GapUp|
- **Edges**: drawn only when |ρ| > 0.40 between two features
- **Edge colour**: red = positive co-movement, blue = negative
- **Edge width** ∝ |ρ|

Tightly connected clusters reveal redundant feature groups.  Isolated nodes are orthogonal to the rest of the feature space and may provide unique information.

In [ ]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

EDGE_THRESH = 0.40

if not _HAS_NX:
    print('networkx not installed — skipping network graph')
else:
    G = nx.Graph()
    rho_target_map = spear_df.set_index('Feature')['rho'].to_dict()
    for feat in ALL_FEATURES:
        G.add_node(feat, group=GROUP_MAP[feat], rho=rho_target_map.get(feat, 0))

    for f1, f2 in combinations(ALL_FEATURES, 2):
        r = corr_matrix.loc[f1, f2]
        if abs(r) >= EDGE_THRESH:
            G.add_edge(f1, f2, weight=abs(r), sign=np.sign(r))

    print(f'Nodes: {G.number_of_nodes()}  |  Edges (|rho|>{EDGE_THRESH}): {G.number_of_edges()}')

    pos          = nx.spring_layout(G, seed=42, k=2.5, iterations=100)
    node_colors  = [GROUP_COLORS[d['group']] for _, d in G.nodes(data=True)]
    node_sizes   = [1200 * abs(d['rho']) + 200   for _, d in G.nodes(data=True)]
    edge_colors  = ['#e74c3c' if d['sign'] > 0 else '#3498db'
                    for _, _, d in G.edges(data=True)]
    edge_widths  = [d['weight'] * 5 for _, _, d in G.edges(data=True)]

    fig, ax = plt.subplots(figsize=(18, 13))
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors,
                           node_size=node_sizes, alpha=0.9)
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color=edge_colors,
                           width=edge_widths, alpha=0.5)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=7.5, font_weight='bold')

    leg_nodes = [Patch(color=c, label=g) for g, c in GROUP_COLORS.items()]
    leg_edges = [
        Line2D([0],[0], color='#e74c3c', lw=2, label=f'Positive correlation (|rho|>{EDGE_THRESH})'),
        Line2D([0],[0], color='#3498db', lw=2, label='Negative correlation'),
    ]
    ax.legend(handles=leg_nodes + leg_edges, loc='lower left', fontsize=8, framealpha=0.9)
    ax.set_title(f'Feature Correlation Network  (nodes sized by |rho with GapUp|, '
                 f'edges for |rho| > {EDGE_THRESH})', fontsize=12)
    ax.axis('off')
    plt.tight_layout()
    plt.show()


---
## 14  Summary — Key Correlation Findings


In [ ]:
# Master ranking table
summary = spear_df[['Feature','Group','rho','pval','Significant']].copy()
summary['Partial rho'] = summary['Feature'].map(partial_rho)
summary['Lag+1 rho']   = summary['Feature'].map(lag_df['lag_+1'])
summary = summary.sort_values('rho', key=abs, ascending=False)
print('Feature Correlation Summary (sorted by |Spearman rho|)')
print(summary.round(4).to_string(index=False))


### Interpretation Guide

**1. Raw Spearman ρ (Section 3)**  
The top-ranked features by |ρ| are the strongest monotone predictors of GapUp.  Features with Bonferroni-significant p-values have correlations that are unlikely to be sampling noise.

**2. Feature-feature collinearity (Section 4)**  
Red clusters in the Spearman matrix identify groups of near-redundant features.  The hierarchical clustering order directly matches the VIF pruning order recommended in `statistical_analysis.ipynb`.

**3. Autocorrelation structure (Section 5)**  
A Durbin-Watson near 2.0 and insignificant Ljung-Box counts for `GapUp` confirm that the target is largely white noise week-to-week — validating the cross-sectional (non-sequential) model approach.  Features with high Ljung-Box counts carry multi-week momentum that LSTM models (`lstm_model.ipynb`) can exploit.

**4. Lead-lag relationships (Section 6)**  
Features peaking at **lag +1** are directly actionable — observable on Friday, predictive of the following week's gap.  Features peaking at **negative lags** are reactors, not predictors.

**5. Temporal drift (Sections 7 & 8)**  
Rolling and inter-annual correlation plots reveal regime instability.  Features whose rolling ρ flips sign across years are unreliable in a static model.  Walk-forward evaluation (used in all project notebooks) partially mitigates this by retraining each fold on recent data only.

**6. Granger causality (Section 10)**  
Features passing Granger causality at lag 1 (stationary, p < 0.05) provide formal evidence that their past values carry predictive precedence over GapUp.  This is a necessary (not sufficient) condition for linear predictive power.

**7. Cross-ticker entanglement (Section 11)**  
Mean pairwise GapUp correlation > 0.10 implies that effective sample size is substantially smaller than the row count.  The rolling synchronisation plot reveals whether entanglement spikes during specific regimes (e.g. rate-hike cycles, earnings seasons).

**8. Partial vs raw ρ (Section 12)**  
Features with large gap between raw and partial correlation are redundant — their information is already encoded in other features.  These are the strongest candidates for feature removal to reduce overfitting risk.

**9. Network graph (Section 13)**  
Isolated nodes in the network are orthogonal to the rest of the feature space — they carry unique information and should be retained even if their raw ρ with GapUp is modest.